# MinHash/LSH Demo


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 160)
plt.style.use("seaborn-v0_8-whitegrid")


def resolve_artifact_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for candidate in candidates:
        artifact_dir = candidate / "jupyter" / "output" / "lsh"
        if artifact_dir.exists():
            return artifact_dir
    raise FileNotFoundError("Could not find jupyter/output/lsh. Run the pipeline scripts first.")


def preview_text(text: object, width: int = 120) -> str:
    compact = " ".join(str(text).split())
    return compact if len(compact) <= width else compact[: width - 1] + "…"


def top_cluster_table(clusters: pd.DataFrame, scale: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    summary = (
        clusters.groupby("cluster_id", as_index=False)["cluster_size"]
        .max()
        .sort_values(["cluster_size", "cluster_id"], ascending=[False, True])
        .head(top_n)
    )
    sample = (
        clusters[["cluster_id", "tweet_id"]]
        .merge(scale[["tweet_id", "norm_text"]], on="tweet_id", how="left")
        .sort_values(["cluster_id", "tweet_id"])
        .groupby("cluster_id", as_index=False)
        .first()
    )
    table = summary.merge(sample, on="cluster_id", how="left")
    table["example_tweet"] = table["norm_text"].map(preview_text)
    return table[["cluster_id", "cluster_size", "tweet_id", "example_tweet"]]


def pair_examples(verified: pd.DataFrame, lookup: pd.DataFrame, mode: str, top_n: int = 5) -> pd.DataFrame:
    merged = verified.merge(
        lookup.add_suffix("_left"),
        left_on="tweet_id_left",
        right_on="tweet_id_left",
        how="left",
    ).merge(
        lookup.add_suffix("_right"),
        left_on="tweet_id_right",
        right_on="tweet_id_right",
        how="left",
    )

    if mode == "exact":
        table = merged[merged["jaccard"] == 1.0].copy()
        table = table.drop_duplicates(subset=["norm_text_left"])
        table = table.sort_values(["tweet_id_left", "tweet_id_right"])
    elif mode == "near":
        table = merged[(merged["jaccard"] < 1.0) & (merged["jaccard"] >= 0.8)].copy()
        table = table.drop_duplicates(subset=["norm_text_left", "norm_text_right"])
        table = table.sort_values(["jaccard", "tweet_id_left"], ascending=[False, True])
    else:
        raise ValueError("mode must be 'exact' or 'near'")

    table["left_text"] = table["text_left"].map(preview_text)
    table["right_text"] = table["text_right"].map(preview_text)
    return table[["tweet_id_left", "tweet_id_right", "jaccard", "left_text", "right_text"]].head(top_n)


def cluster_samples(cluster_id: int, clusters: pd.DataFrame, scale: pd.DataFrame, top_n: int = 8) -> pd.DataFrame:
    table = (
        clusters[clusters["cluster_id"] == cluster_id]
        .merge(scale[["tweet_id", "text"]], on="tweet_id", how="left")
        .sort_values("tweet_id")
        .head(top_n)
        .copy()
    )
    table["text_preview"] = table["text"].map(lambda value: preview_text(value, width=140))
    return table[["cluster_id", "cluster_size", "tweet_id", "text_preview"]]


ARTIFACT_DIR = resolve_artifact_dir()
FIGURE_DIR = ARTIFACT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

METRICS = json.loads((ARTIFACT_DIR / "metrics.json").read_text(encoding="utf-8"))
SCALE = pd.read_parquet(ARTIFACT_DIR / "scale_shingles.parquet")
VERIFIED = pd.read_parquet(ARTIFACT_DIR / "verified_pairs.parquet")
CLUSTERS = pd.read_parquet(ARTIFACT_DIR / "clusters.parquet")
LOOKUP = SCALE[["tweet_id", "text", "norm_text"]].copy()

CONFIG_RANKING = pd.DataFrame(METRICS["run_lsh"]["config_results"])
CLUSTER_SIZE_DF = (
    CLUSTERS.groupby("cluster_id", as_index=False)["cluster_size"]
    .max()
    .sort_values(["cluster_size", "cluster_id"], ascending=[False, True])
)
TOP_CLUSTERS = top_cluster_table(CLUSTERS, SCALE, top_n=10)
EXACT_EXAMPLES = pair_examples(VERIFIED, LOOKUP, mode="exact", top_n=5)
NEAR_EXAMPLES = pair_examples(VERIFIED, LOOKUP, mode="near", top_n=5)
SELECTED = METRICS["run_lsh"]["selected_config"]
LARGEST_CLUSTER_ID = int(TOP_CLUSTERS.iloc[0]["cluster_id"])

ARTIFACT_DIR

: 

## 1. Executive Summary

In [ ]:
singleton_clusters = int((CLUSTER_SIZE_DF["cluster_size"] == 1).sum())
repeat_clusters = int(len(CLUSTER_SIZE_DF) - singleton_clusters)

overview_df = pd.DataFrame(
    [
        {"metric": "Baseline subset", "value": f"{METRICS['extract_subsets']['baseline_rows']:,} tweets"},
        {"metric": "Scale subset", "value": f"{METRICS['extract_subsets']['scale_rows']:,} tweets"},
        {"metric": "Rows kept after shingles", "value": f"{METRICS['build_shingles']['scale_shingles']['kept_rows']:,} tweets"},
        {"metric": "Baseline exact positives", "value": f"{METRICS['baseline']['positive_pairs']:,} pairs"},
        {"metric": "LSH candidates", "value": f"{METRICS['run_lsh']['scale_run']['candidate_pairs']:,} pairs"},
        {"metric": "Verified near-duplicates", "value": f"{METRICS['verify_and_cluster']['verified_pairs']:,} pairs"},
        {"metric": "Total clusters", "value": f"{METRICS['verify_and_cluster']['clusters']:,}"},
        {"metric": "Repeated-content clusters", "value": f"{repeat_clusters:,}"},
        {"metric": "Largest cluster", "value": f"{METRICS['verify_and_cluster']['largest_cluster_size']:,} tweets"},
        {"metric": "Winning config", "value": SELECTED['config_name']},
    ]
)

display(overview_df)

## 2. Dashboard

In [ ]:
runtime_df = pd.DataFrame(
    [
        {"stage": "Baseline exact", "runtime_seconds": METRICS['baseline']['runtime_seconds']},
        {"stage": "LSH scale run", "runtime_seconds": METRICS['run_lsh']['scale_run']['runtime_seconds']},
    ]
)
funnel_df = pd.DataFrame(
    [
        {"stage": "Scale docs", "count": METRICS['build_shingles']['scale_shingles']['kept_rows']},
        {"stage": "Candidate pairs", "count": METRICS['run_lsh']['scale_run']['candidate_pairs']},
        {"stage": "Verified pairs", "count": METRICS['verify_and_cluster']['verified_pairs']},
    ]
)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].bar(runtime_df['stage'], runtime_df['runtime_seconds'], color=['#ef4444', '#2563eb'])
axes[0, 0].set_title('Runtime comparison')
axes[0, 0].set_ylabel('Seconds')
for idx, value in enumerate(runtime_df['runtime_seconds']):
    axes[0, 0].text(idx, value, f'{value:.2f}', ha='center', va='bottom')

axes[0, 1].bar(funnel_df['stage'], funnel_df['count'], color=['#10b981', '#2563eb', '#7c3aed'])
axes[0, 1].set_title('Duplicate-detection funnel')
axes[0, 1].set_ylabel('Count')
for idx, value in enumerate(funnel_df['count']):
    axes[0, 1].text(idx, value, f'{int(value):,}', ha='center', va='bottom')

config_plot = CONFIG_RANKING.copy()
axes[1, 0].scatter(
    config_plot['candidate_pairs'],
    config_plot['precision'],
    s=150,
    c=config_plot['recall'],
    cmap='viridis',
    edgecolors='black',
)
for row in config_plot.itertuples(index=False):
    axes[1, 0].annotate(row.config_name, (row.candidate_pairs, row.precision), xytext=(6, 6), textcoords='offset points', fontsize=9)
axes[1, 0].set_title('Config trade-off: fewer candidates vs precision')
axes[1, 0].set_xlabel('Candidate pairs on baseline')
axes[1, 0].set_ylabel('Precision')

cluster_hist = CLUSTER_SIZE_DF['cluster_size']
bins = [1, 2, 3, 5, 10, 20, 40, 60, 80]
axes[1, 1].hist(cluster_hist, bins=bins, color='#f59e0b', edgecolor='black')
axes[1, 1].set_title('Cluster size distribution')
axes[1, 1].set_xlabel('Cluster size')
axes[1, 1].set_ylabel('Number of clusters')

for ax in axes.flat:
    ax.tick_params(axis='x', rotation=10)

plt.tight_layout()
fig.savefig(FIGURE_DIR / 'dashboard.png', dpi=160, bbox_inches='tight')
plt.show()

## 3. Vì sao chọn config này

In [ ]:
selection_view = CONFIG_RANKING[
    ['config_name', 'shingle_size', 'num_perm', 'bands', 'rows', 'candidate_pairs', 'precision', 'recall']
].copy()
selection_view['selected'] = selection_view['config_name'].eq(SELECTED['config_name'])

display(selection_view)

display(Markdown(
    f"**Config thắng:** `{SELECTED['config_name']}`  \n"
    f"- Recall trên baseline = **{CONFIG_RANKING.iloc[0]['recall']:.3f}**  \n"
    f"- Candidate pairs trên baseline = **{int(CONFIG_RANKING.iloc[0]['candidate_pairs']):,}**  \n"
    f"- Precision trên baseline = **{CONFIG_RANKING.iloc[0]['precision']:.3f}**  \n"
    "- Đây là config ít candidate nhất trong nhóm đạt recall 1.0 nên phù hợp làm cấu hình demo/final."
))

## 4. Top cluster lớn nhất

In [ ]:
display(TOP_CLUSTERS)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = TOP_CLUSTERS.iloc[::-1]
ax.barh(plot_df['cluster_id'].astype(str), plot_df['cluster_size'], color='#2563eb')
ax.set_title('Top 10 clusters by size')
ax.set_xlabel('Cluster size')
ax.set_ylabel('Cluster ID')
for idx, value in enumerate(plot_df['cluster_size']):
    ax.text(value, idx, f' {int(value)}', va='center')
plt.tight_layout()
fig.savefig(FIGURE_DIR / 'top_clusters.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
largest_cluster_preview = cluster_samples(LARGEST_CLUSTER_ID, CLUSTERS, SCALE, top_n=8)
display(Markdown(f"**Largest cluster:** `{LARGEST_CLUSTER_ID}` with **{int(TOP_CLUSTERS.iloc[0]['cluster_size'])}** tweets"))
display(largest_cluster_preview)

## 5. Exact duplicate tiêu biểu

In [ ]:
display(EXACT_EXAMPLES)

## 6. Near duplicate tiêu biểu

In [ ]:
display(NEAR_EXAMPLES)

## 7. Talking points gợi ý

In [ ]:
talking_points = [
    f"Từ 50,000 tweet trong scale subset, pipeline giữ lại {METRICS['build_shingles']['scale_shingles']['kept_rows']:,} tweet đủ dài để so sánh bằng shingle.",
    f"Baseline exact trên 3,000 tweet tìm được {METRICS['baseline']['positive_pairs']:,} cặp near-duplicate và dùng làm ground truth để tune LSH.",
    f"Config {SELECTED['config_name']} được chọn vì đạt recall 1.0 trên baseline nhưng có số candidate thấp nhất trong nhóm recall tối đa.",
    f"Trên 49,852 tweet đã shingled, LSH rút bài toán về {METRICS['run_lsh']['scale_run']['candidate_pairs']:,} candidate pairs, rồi verify exact giữ lại {METRICS['verify_and_cluster']['verified_pairs']:,} cặp thật sự gần nhau.",
    f"Kết quả cuối tạo ra {METRICS['verify_and_cluster']['clusters']:,} cụm; cụm lớn nhất có {METRICS['verify_and_cluster']['largest_cluster_size']:,} tweet, cho thấy hiện tượng lặp nội dung là có thật và khá rõ.",
    "Ví dụ duplicate không chỉ là bản sao y hệt, mà còn có các biến thể thay hashtag, account mention, link, hoặc chỉnh nhẹ câu chữ nhưng vẫn cùng nội dung.",
]

display(Markdown("\n".join([f"- {point}" for point in talking_points])))